In [4]:
import ollama
import gradio as gr

SYSTEM = ("You are an OS and shell tutor. Answer in at most 5-6 sentences or a short list. "
          "Be concise and accurate. Give a short example only if needed. "
          "Only give a detailed answer if and when asked, also include an example with it."
          "End by offering to explain more if the student wants.")

MAX_HISTORY = 6   # keep only the last 6 messages

def to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
    return str(content)

def chat(message, history):
    msgs = []
    for turn in history:
        if isinstance(turn, dict):
            msgs.append({"role": turn["role"], "content": to_text(turn["content"])})
        else:
            user, bot = turn
            msgs.append({"role": "user", "content": to_text(user)})
            if bot:
                msgs.append({"role": "assistant", "content": to_text(bot)})
    msgs = msgs[-MAX_HISTORY:]
    msgs = [{"role": "system", "content": SYSTEM}] + msgs
    msgs.append({"role": "user", "content": to_text(message)})

    partial = ""
    for chunk in ollama.chat(
        model="os-tutor",
        messages=msgs,
        stream=True,
        keep_alive="30m",
        options={"num_predict": 300, "num_ctx": 2048,
                 "temperature": 0.7, "top_p": 0.8, "top_k": 20},
    ):
        partial += chunk["message"]["content"]
        yield partial

gr.ChatInterface(chat, title="OS & Shell Tutor").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [2]:
import ollama
import gradio as gr

SYSTEM = ("You are an OS and shell tutor. Answer in at most 5-6 sentences or a short list. "
          "Be concise and accurate. Give a short example only if needed. "
          "End by offering to explain more if the student wants.")

def to_text(content):
    # newer Gradio can send content as a list of parts instead of a string
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
    return str(content)

def chat(message, history):
    msgs = [{"role": "system", "content": SYSTEM}]
    for turn in history:
        if isinstance(turn, dict):                 # new format
            msgs.append({"role": turn["role"], "content": to_text(turn["content"])})
        else:                                      # old format: (user, bot)
            user, bot = turn
            msgs.append({"role": "user", "content": to_text(user)})
            if bot:
                msgs.append({"role": "assistant", "content": to_text(bot)})
    msgs.append({"role": "user", "content": to_text(message)})

    partial = ""
    for chunk in ollama.chat(model="os-tutor", messages=msgs, stream=True):
        partial += chunk["message"]["content"]
        yield partial                              # streams the answer as it is generated

gr.ChatInterface(chat, title="OS & Shell Tutor").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
